# LocalTransform 数据处理教程

## 概述

本 notebook 是 LocalTransform 教程系列的第 2 部分，聚焦于**从原始反应 SMILES 到模型输入**的完整数据处理流程。

> 前置阅读: Notebook 1 — 环境配置  
> 后续阅读: Notebook 3 — 模型展示

### 本 notebook 内容

| 章节 | 内容 |
|------|------|
| 1 | 原始数据格式与整体数据流 |
| 2 | Step 1: 反应模板提取（Extract Templates） |
| 3 | Step 2: 反应标注（Label Reactions） |
| 4 | Step 3: 分子图构建 — 特征图（Feature Graph） |
| 5 | Step 4: 分子图构建 — 稠密图（Dense Graph） |
| 6 | Step 5: 批处理与数据整理（Collation） |
| 7 | 总结与下一步 |

### 教学策略

- 使用 **1-2 个反应作为最小样本** 贯穿全流程
- 每一步都映射回源码中的文件和函数
- 用 `print()` 展示中间数据结构，确保每一步可观察

---

In [1]:
# ====== 环境初始化 ======
import os, sys
from pathlib import Path

def find_project_root(start=None):
    here = Path(start or os.getcwd()).resolve()
    if here.is_file():
        here = here.parent
    for candidate in [here, *here.parents]:
        if (candidate / 'teaching_demos').exists():
            return candidate
    raise FileNotFoundError('无法定位项目根目录')

PROJECT_ROOT = find_project_root()
SOURCE_REPO = PROJECT_ROOT / 'source_repos' / 'LocalTransform'
EXPECTED_PYTHON = PROJECT_ROOT / 'envs' / 'localtransform_tutorial_envs' / 'bin' / 'python'

for path in [SOURCE_REPO, SOURCE_REPO / 'scripts']:
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

import numpy as np
import pandas as pd
from rdkit import Chem, RDLogger
RDLogger.DisableLog('rdApp.*')  # 抑制 RDKit 警告

print(f'项目根目录: {PROJECT_ROOT}')
print(f'源码目录: {SOURCE_REPO}')
print(f'当前 Python: {sys.executable}')
if EXPECTED_PYTHON.exists() and Path(sys.executable).resolve() != EXPECTED_PYTHON.resolve():
    print('警告: 当前 kernel 不是 localtransform_tutorial_envs，若后续导入失败请切换内核。')

项目根目录: /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis
源码目录: /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/source_repos/LocalTransform
当前 Python: /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/envs/localtransform_tutorial_envs/bin/python


## 1. 原始数据格式与整体数据流

### 原始数据

LocalTransform 使用 **USPTO_480k** 数据集（约 48 万条有机化学反应）。原始格式为每行一条反应 SMILES：

```
[反应物]>>[产物]
```

### 完整数据流

```
原始反应 SMILES (train/valid/test.txt)
    │
    ├── [Phase 1] 模板提取 (Extract_from_train_data.py)
    │     ├── 输入: train.txt
    │     ├── 输出: real_templates.csv, virtual_templates.csv, template_infos.csv
    │     └── 作用: 从训练集中统计所有局部反应模板
    │
    ├── [Phase 2] 数据标注 (Run_preprocessing.py)
    │     ├── 输入: train/valid/test.txt + 模板表
    │     ├── 输出: labeled_data.csv
    │     └── 作用: 为每个反应的每条键分配模板类别标签
    │
    └── [Phase 3] 图构建 (dataset.py)
          ├── 特征图: SMILES → DGL图 (原子+键特征)
          ├── 稠密图: 原子距离矩阵 + 虚拟键/真实键列表
          └── 标签: 每条键的模板类别
```

### 源码对应

| Phase | 文件 | 核心函数 |
|-------|------|----------|
| 模板提取 | `preprocessing/Extract_from_train_data.py` | `extract_templates()`, `export_template()` |
| 数据标注 | `preprocessing/Run_preprocessing.py` | `labeling_dataset()`, `combine_preprocessed_data()` |
| 图构建 | `scripts/dataset.py` | `USPTODataset`, `get_bonds()`, `get_adm()` |
| 模板底层 | `LocalTemplate/template_extractor.py` | `extract_from_reaction()` |

## 2. Step 1: 反应模板提取

### 为什么需要这一步？

LocalTransform 是**模板驱动**的方法。它不直接生成产物 SMILES，而是预测每条键上应该应用哪个"局部模板"。因此需要先从训练数据中提取所有可能的局部模板。

### 核心思想

对于每条反应 `反应物 >> 产物`：
1. 比较反应物和产物中的每个原子，找到**发生变化的原子**（反应中心）
2. 提取反应中心周围的**局部 SMARTS 模板**
3. 将模板分为两类：
   - **真实键模板 (Real)**: 涉及键断裂(B)、键变化(C)、原子移除(R)
   - **虚拟键模板 (Virtual)**: 涉及键形成(A)

### 源码对应
- 文件: `preprocessing/Extract_from_train_data.py`
- 函数: `extract_templates()`, `get_reaction_template()`
- 模板提取器: `LocalTemplate/template_extractor.py` → `extract_from_reaction()`

In [2]:
# ====== Step 2.1: 构建模板提取器 ======
# 源码对应: preprocessing/Extract_from_train_data.py → build_template_extractor()

from LocalTemplate.template_extractor import extract_from_reaction

# 提取器设置
setting = {
    'verbose': False,       # 不打印调试信息
    'use_stereo': False,    # 不使用立体化学（简化教学）
    'use_symbol': False,    # 使用通配符 [A] 而非具体原子符号
    'max_unmap': 5,         # 允许最多 5 个未映射原子
    'retro': False,         # 正向合成（非逆合成）
    'remote': True,         # 考虑远程变化（电荷、H数）
    'least_atom_num': 2,    # 模板至少包含 2 个原子
}

extractor = lambda x: extract_from_reaction(x, setting)
print('模板提取器设置:', setting)

模板提取器设置: {'verbose': False, 'use_stereo': False, 'use_symbol': False, 'max_unmap': 5, 'retro': False, 'remote': True, 'least_atom_num': 2}


In [3]:
# ====== Step 2.2: 用一条反应演示模板提取 ======
# 示例: Suzuki 偶联反应（简化版本）
# 苯基硼酸 + 溴苯 → 联苯 (原子映射已手动标注)

reaction_smiles = '[CH3:1][C:2](=[O:3])[Cl:4].[NH2:5][CH3:6]>>[CH3:1][C:2](=[O:3])[NH:5][CH3:6]'

print(f'反应 SMILES: {reaction_smiles}')
print(f'含义: 乙酰氯 + 甲胺 → N-甲基乙酰胺 + HCl')

# 解析反应
rxn = {
    'reactants': reaction_smiles.split('>>')[0],
    'products': reaction_smiles.split('>>')[1],
    '_id': 0
}

# 提取模板
result = extractor(rxn)

print(f'\n提取结果:')
print(f'  反应物 (标准化): {result.get("reactants", "N/A")}')
print(f'  产物 (标准化): {result.get("products", "N/A")}')
print(f'  反应 SMARTS 模板: {result.get("reaction_smarts", "N/A")}')
print(f'  编辑信息 (edits): {result.get("edits", "N/A")}')
print(f'  H变化: {result.get("H_change", "N/A")}')
print(f'  电荷变化: {result.get("Charge_change", "N/A")}')
print(f'  手性变化: {result.get("Chiral_change", "N/A")}')
print(f'  必要试剂: {result.get("necessary_reagent", "N/A")}')

反应 SMILES: [CH3:1][C:2](=[O:3])[Cl:4].[NH2:5][CH3:6]>>[CH3:1][C:2](=[O:3])[NH:5][CH3:6]
含义: 乙酰氯 + 甲胺 → N-甲基乙酰胺 + HCl

提取结果:
  反应物 (标准化): [CH3:1][C:2](=[O:3])[Cl:7].[NH2:5][CH3:6]
  产物 (标准化): [CH3:1][C:2](=[O:3])[NH:5][CH3:6]
  反应 SMARTS 模板: [A:1].[A:2]-[A:3]>>[A:1]-[A:2]
  编辑信息 (edits): {'A': ([(4, 1)], [(5, 2)], [(1, 2)]), 'B': ([(1, 3)], [(2, 7)], [(2, 3)]), 'C': ([], [], []), 'R': ([], [], [])}
  H变化: {2: 0, 1: -1}
  电荷变化: {2: 0, 1: 0}
  手性变化: {2: 0, 1: 0}
  必要试剂: []


In [4]:
# ====== Step 2.3: 解读模板结构 ======
# 深入分析提取的模板各组成部分

if 'reaction_smarts' in result and 'edits' in result:
    template = result['reaction_smarts']
    edits = result['edits']
    H_change = result['H_change']
    Charge_change = result['Charge_change']
    Chiral_change = result['Chiral_change']

    print('=== 模板结构解析 ===')
    print(f'\n1. 反应 SMARTS: {template}')
    print(f'   箭头左边 = 反应物模式, 箭头右边 = 产物模式')
    print(f'   [A:n] 表示通配原子，:n 是映射编号')
    
    print(f'\n2. 编辑类型 (edits):')
    for edit_type, edit_info in edits.items():
        type_names = {'A': '键形成(Addition)', 'B': '键断裂(Break)', 'C': '键变化(Change)', 'R': '原子移除(Remove)'}
        name = type_names.get(edit_type, edit_type)
        bonds = edit_info[0]  # 涉及的键
        print(f'   {edit_type} ({name}):')
        print(f'     涉及的键: {bonds}')
        if len(edit_info) > 2:
            print(f'     模板片段: {edit_info[2]}')
    
    print(f'\n3. 原子属性变化:')
    print(f'   H 数变化: {H_change}')
    print(f'   电荷变化: {Charge_change}')
    print(f'   手性变化: {Chiral_change}')
    
    # 构建完整模板字符串（与源码一致）
    H_code = ''.join([str(H_change[k+1]) for k in range(len(H_change))])
    C_code = ''.join([str(Charge_change[k+1]) for k in range(len(Charge_change))])
    S_code = ''.join([str(Chiral_change[k+1]) for k in range(len(Chiral_change))])
    full_template = '_'.join([template, H_code, C_code, S_code])
    
    print(f'\n4. 组合后的完整模板字符串:')
    print(f'   {full_template}')
    print(f'   格式: SMARTS_H变化码_电荷变化码_手性变化码')
else:
    print('模板提取失败，请检查反应 SMILES')

=== 模板结构解析 ===

1. 反应 SMARTS: [A:1].[A:2]-[A:3]>>[A:1]-[A:2]
   箭头左边 = 反应物模式, 箭头右边 = 产物模式
   [A:n] 表示通配原子，:n 是映射编号

2. 编辑类型 (edits):
   A (键形成(Addition)):
     涉及的键: [(4, 1)]
     模板片段: [(1, 2)]
   B (键断裂(Break)):
     涉及的键: [(1, 3)]
     模板片段: [(2, 3)]
   C (键变化(Change)):
     涉及的键: []
     模板片段: []
   R (原子移除(Remove)):
     涉及的键: []
     模板片段: []

3. 原子属性变化:
   H 数变化: {2: 0, 1: -1}
   电荷变化: {2: 0, 1: 0}
   手性变化: {2: 0, 1: 0}

4. 组合后的完整模板字符串:
   [A:1].[A:2]-[A:3]>>[A:1]-[A:2]_-10_00_00
   格式: SMARTS_H变化码_电荷变化码_手性变化码


In [5]:
# ====== Step 2.4: 理解模板分类 — 虚拟键 vs 真实键 ======
# 源码对应: preprocessing/Extract_from_train_data.py → extract_templates() 中的分类逻辑

if 'edits' in result:
    edits = result['edits']
    
    print('=== 模板分类规则 ===')
    print()
    print('LocalTransform 将每个编辑操作分为两大类：')
    print()
    
    for edit_type, edit_info in edits.items():
        bonds = edit_info[0]
        if len(bonds) > 0:
            if edit_type == 'A':
                category = '虚拟键模板 (Virtual Template)'
                reason = '键形成 → 原本不存在的键（虚拟键）上发生反应'
            else:
                category = '真实键模板 (Real Template)'
                reason = f'键断裂/变化/移除 → 已存在的键（真实键）上发生反应'
            
            full_template_key = f'{full_template}_{edit_type}'
            print(f'编辑类型 {edit_type}: → {category}')
            print(f'  原因: {reason}')
            print(f'  涉及的原子对: {bonds}')
            print(f'  完整模板键: {full_template_key}')
            print()

    print('> 这就是为什么模型有两套分类头（output_v 和 output_r）：')
    print('> 虚拟键用 virtual template 分类器，真实键用 real template 分类器')

=== 模板分类规则 ===

LocalTransform 将每个编辑操作分为两大类：

编辑类型 A: → 虚拟键模板 (Virtual Template)
  原因: 键形成 → 原本不存在的键（虚拟键）上发生反应
  涉及的原子对: [(4, 1)]
  完整模板键: [A:1].[A:2]-[A:3]>>[A:1]-[A:2]_-10_00_00_A

编辑类型 B: → 真实键模板 (Real Template)
  原因: 键断裂/变化/移除 → 已存在的键（真实键）上发生反应
  涉及的原子对: [(1, 3)]
  完整模板键: [A:1].[A:2]-[A:3]>>[A:1]-[A:2]_-10_00_00_B

> 这就是为什么模型有两套分类头（output_v 和 output_r）：
> 虚拟键用 virtual template 分类器，真实键用 real template 分类器


In [6]:
# ====== Step 2.5: 查看已提取的模板库 ======
# 源码已经在 USPTO_480k 数据集上提取好了模板，我们直接查看

data_dir = SOURCE_REPO / 'data' / 'USPTO_480k'

real_df = pd.read_csv(data_dir / 'real_templates.csv')
virtual_df = pd.read_csv(data_dir / 'virtual_templates.csv')
info_df = pd.read_csv(data_dir / 'template_infos.csv')

print('=== 已提取的模板库统计 ===')
print(f'真实键模板数: {len(real_df)}')
print(f'虚拟键模板数: {len(virtual_df)}')
print(f'全局模板数: {len(info_df)}')

print(f'\n最常见的 5 个真实键模板:')
top_real = real_df.nlargest(5, 'Frequency')
for _, row in top_real.iterrows():
    t = row['Template']
    parts = t.rsplit('_', 4)
    smarts = parts[0] if len(parts) >= 4 else t
    print(f'  Class {row["Class"]}: freq={row["Frequency"]:>5}  {smarts[:80]}')

print(f'\n最常见的 5 个虚拟键模板:')
top_virtual = virtual_df.nlargest(5, 'Frequency')
for _, row in top_virtual.iterrows():
    t = row['Template']
    parts = t.rsplit('_', 4)
    smarts = parts[0] if len(parts) >= 4 else t
    print(f'  Class {row["Class"]}: freq={row["Frequency"]:>5}  {smarts[:80]}')

=== 已提取的模板库统计 ===
真实键模板数: 4370
虚拟键模板数: 2535
全局模板数: 2846

最常见的 5 个真实键模板:
  Class 4370: freq=157844  [A:1].[A:2]-[A:3]>>[A:1]-[A:2]
  Class 4369: freq=46987  [A:1]-[A:2]>>[A:1]
  Class 4368: freq=28122  [A:1]-[A:3].[A:2]-[A:4]>>[A:1]-[A:2]
  Class 4366: freq=19248  [A:1].[A:3]-[A:2]=[A:4]>>[A:1]-[A:2]=[A:3]
  Class 4367: freq=19248  [A:1].[A:3]-[A:2]=[A:4]>>[A:1]-[A:2]=[A:3]

最常见的 5 个虚拟键模板:
  Class 2535: freq=157844  [A:1].[A:2]-[A:3]>>[A:1]-[A:2]
  Class 2534: freq=28122  [A:1]-[A:3].[A:2]-[A:4]>>[A:1]-[A:2]
  Class 2533: freq=19248  [A:1].[A:3]-[A:2]=[A:4]>>[A:1]-[A:2]=[A:3]
  Class 2532: freq=10218  [A:1].[A:2]=[A:3]>>[A:1]-[A:2]
  Class 2531: freq= 9947  [A:1].[A:2]=[A:3]>>[A:1]-[A:2]-[A:3]


## 3. Step 2: 反应标注

### 为什么需要这一步？

模板提取完成后，需要为训练集中的每条反应生成**键级别的标签**：
- 每条真实键→ 赋予一个 real template class（0 表示该键不参与反应）
- 每条虚拟键→ 赋予一个 virtual template class（0 表示该键不形成）

这也是 LocalTransform 与全局模板方法的核心区别：标签是**逐键**的，而非**逐分子**的。

### 源码对应
- 文件: `preprocessing/Run_preprocessing.py`
- 函数: `labeling_dataset()`, `get_edit_site()`
- 核心逻辑: 遍历每条反应 → 提取模板 → 找到涉及的键 → 在键列表中定位索引 → 赋予模板类别

In [7]:
# ====== Step 3.1: 获取键列表（编辑位点）======
# 源码对应: preprocessing/Run_preprocessing.py → get_edit_site()
# 与 scripts/dataset.py → get_bonds() 逻辑一致

def get_edit_site(smiles):
    """获取分子中所有虚拟键和真实键的列表
    
    Args:
        smiles: 分子 SMILES
    Returns:
        V: 虚拟键列表 [(a,b), ...] — 未成键的原子对
        B: 真实键列表 [(a,b), ...] — 已有的化学键
    """
    mol = Chem.MolFromSmiles(smiles)
    A = list(range(mol.GetNumAtoms()))
    B = []
    for atom in mol.GetAtoms():
        for bond in atom.GetBonds():
            begin = bond.GetBeginAtom().GetIdx()
            end = bond.GetEndAtom().GetIdx()
            other = end if begin == atom.GetIdx() else begin
            B.append((atom.GetIdx(), other))
    V = [(a, b) for a in A for b in A if a != b and (a, b) not in B]
    return V, B

# 用乙酰氯做示例
reactant_smiles = 'CC(=O)Cl'
mol = Chem.MolFromSmiles(reactant_smiles)
print(f'分子: {reactant_smiles}')
print(f'原子: {[(a.GetIdx(), a.GetSymbol()) for a in mol.GetAtoms()]}')

V, B = get_edit_site(reactant_smiles)
print(f'\n真实键 (Real bonds): {len(B)} 条')
for i, (a, b) in enumerate(B[:10]):
    sym_a = mol.GetAtomWithIdx(a).GetSymbol()
    sym_b = mol.GetAtomWithIdx(b).GetSymbol()
    print(f'  [{i}] {sym_a}({a}) — {sym_b}({b})')

print(f'\n虚拟键 (Virtual bonds): {len(V)} 条')
for i, (a, b) in enumerate(V[:10]):
    sym_a = mol.GetAtomWithIdx(a).GetSymbol()
    sym_b = mol.GetAtomWithIdx(b).GetSymbol()
    print(f'  [{i}] {sym_a}({a}) ··· {sym_b}({b})')
if len(V) > 10:
    print(f'  ... (共 {len(V)} 条)')

分子: CC(=O)Cl
原子: [(0, 'C'), (1, 'C'), (2, 'O'), (3, 'Cl')]

真实键 (Real bonds): 6 条
  [0] C(0) — C(1)
  [1] C(1) — C(0)
  [2] C(1) — O(2)
  [3] C(1) — Cl(3)
  [4] O(2) — C(1)
  [5] Cl(3) — C(1)

虚拟键 (Virtual bonds): 6 条
  [0] C(0) ··· O(2)
  [1] C(0) ··· Cl(3)
  [2] O(2) ··· C(0)
  [3] O(2) ··· Cl(3)
  [4] Cl(3) ··· C(0)
  [5] Cl(3) ··· O(2)


In [8]:
# ====== Step 3.2: 为一条反应生成键级别标签 ======
# 源码对应: preprocessing/Run_preprocessing.py → labeling_dataset() 中的核心循环

# 使用之前提取的模板结果
reaction_smiles = '[CH3:1][C:2](=[O:3])[Cl:4].[NH2:5][CH3:6]>>[CH3:1][C:2](=[O:3])[NH:5][CH3:6]'
rxn = {'reactants': reaction_smiles.split('>>')[0], 'products': reaction_smiles.split('>>')[1], '_id': 0}
result = extractor(rxn)

if 'edits' in result:
    reactant = result['reactants']
    edits = result['edits']
    
    print(f'标准化反应物: {reactant}')
    
    # 获取键列表
    virtual_sites, real_sites = get_edit_site(reactant)
    print(f'虚拟键数: {len(virtual_sites)}, 真实键数: {len(real_sites)}')
    
    # 为每条键匹配标签
    edit_bonds = {edit_type: edit_bond[0] for edit_type, edit_bond in edits.items()}
    
    # 初始化标签（全部为 0 = 不参与反应）
    v_labels = [0] * len(virtual_sites)
    r_labels = [0] * len(real_sites)
    
    print(f'\n=== 标签分配过程 ===')
    for edit_type, bonds in edit_bonds.items():
        for bond in bonds:
            if edit_type == 'A':  # 键形成 → 虚拟键
                try:
                    idx = virtual_sites.index(bond)
                    v_labels[idx] = 1  # 这里简化为1，实际是模板类别ID
                    print(f'  键形成: 原子对 {bond} → 虚拟键索引 {idx} → 标签 = 模板类别')
                except ValueError:
                    print(f'  键形成: 原子对 {bond} 未在虚拟键列表中找到')
            else:  # 键断裂/变化/移除 → 真实键
                try:
                    idx = real_sites.index(bond)
                    r_labels[idx] = 1
                    print(f'  键{edit_type}: 原子对 {bond} → 真实键索引 {idx} → 标签 = 模板类别')
                except ValueError:
                    print(f'  键{edit_type}: 原子对 {bond} 未在真实键列表中找到')
    
    reactive_v = sum(1 for l in v_labels if l > 0)
    reactive_r = sum(1 for l in r_labels if l > 0)
    print(f'\n标签统计:')
    print(f'  反应性虚拟键: {reactive_v}/{len(v_labels)}')
    print(f'  反应性真实键: {reactive_r}/{len(r_labels)}')
    print(f'  非反应性键: {len(v_labels) + len(r_labels) - reactive_v - reactive_r}')
    print(f'\n> 绝大多数键的标签为 0（不参与反应），这就是为什么使用加权损失函数')

标准化反应物: [CH3:1][C:2](=[O:3])[Cl:7].[NH2:5][CH3:6]
虚拟键数: 22, 真实键数: 8

=== 标签分配过程 ===
  键形成: 原子对 (4, 1) → 虚拟键索引 15 → 标签 = 模板类别
  键B: 原子对 (1, 3) → 真实键索引 3 → 标签 = 模板类别

标签统计:
  反应性虚拟键: 1/22
  反应性真实键: 1/8
  非反应性键: 28

> 绝大多数键的标签为 0（不参与反应），这就是为什么使用加权损失函数


In [9]:
# ====== Step 3.3: 查看预处理好的标注数据 ======
# 源码已在 USPTO_480k 上预处理好了标注数据

test_df = pd.read_csv(SOURCE_REPO / 'data' / 'USPTO_480k' / 'preprocessed_test.csv', nrows=5)

print('预处理后的数据格式 (preprocessed_test.csv 前5行):')
print(f'列名: {list(test_df.columns)}')
print()

for i, row in test_df.iterrows():
    print(f'--- 反应 #{i} ---')
    # 只显示 SMILES 的前60个字符
    r = str(row['Reactants'])[:60]
    p = str(row.get('Products', 'N/A'))[:60]
    print(f'  反应物: {r}...' if len(str(row['Reactants'])) > 60 else f'  反应物: {r}')
    print(f'  试剂: {row.get("Reagents", "nan")}'[:60])
    
    # 解析标签
    labels = str(row.get('Labels_sep', '[]'))[:100]
    print(f'  标签(sep): {labels}...' if len(str(row.get('Labels_sep', ''))) > 100 else f'  标签(sep): {labels}')
    print(f'  频率: {row.get("Frequency", "N/A")}')
    print()

print('> Labels_sep 格式: [(键类型, 键索引, 模板类别), ...]')
print('>   键类型: v=虚拟键, r=真实键')
print('>   键索引: 在虚拟/真实键列表中的位置')
print('>   模板类别: 模板分类器的类别ID（0=不反应）')

预处理后的数据格式 (preprocessed_test.csv 前5行):
列名: ['Unnamed: 0', 'Reactants', 'Reagents', 'Products', 'Labels_sep', 'Labels_mix', 'Frequency']

--- 反应 #0 ---
  反应物: [c:2]1([F:21])[c:3]([N+:10](=[O:11])[O-:12])[cH:4][c:5]([F:9...
  试剂: C1CCOC1.[H-].[Na+]
  标签(sep): [('v', 204, 2535), ('r', 0, 4370)]
  频率: 157844

--- 反应 #1 ---
  反应物: [CH3:1][c:2]1[n:3][n:4](-[c:11]2[cH:12][cH:13][c:14]([C:17](...
  试剂: CCCCP(CCCC)CCCC.Cc1ccccc1
  标签(sep): [('v', 618, 2535), ('r', 65, 4370)]
  频率: 157844

--- 反应 #2 ---
  反应物: [NH2:1][CH:2]1[CH:3]([OH:23])[CH2:4][N:5]([CH2:8][CH:9]2[CH2...
  试剂: CCOCC.ClC(Cl)Cl.Cl.Cl
  标签(sep): [('v', 28, 2532), ('r', 69, 4365)]
  频率: 10218

--- 反应 #3 ---
  反应物: [OH:1][c:2]1[cH:3][cH:4][c:5]([N:8]2[C:9](=[O:31])[C:10](=[C...
  试剂: CCN(CC)CC.ClCCl
  标签(sep): [('v', 29, 2535), ('r', 70, 4370)]
  频率: 157844

--- 反应 #4 ---
  反应物: [Cl:11][c:12]1[c:13]([CH2:20][O:21][CH:22]2[O:23][CH2:24][CH...
  试剂: OCc1c(Cl)cncc1Cl
  标签(sep): [('r', 28, 4362), ('r', 29, 4362)]
  频率: 8896

> Labels_s

## 4. Step 3: 分子图构建 — 特征图 (Feature Graph)

### 为什么需要特征图？

模型的 MPNN 消息传递需要在分子图上进行。特征图捕获分子的**拓扑结构 + 原子化学性质 + 键类型信息**。

### 核心思想

使用 DGL-Life 提供的特征化工具：
- **WeaveAtomFeaturizer**: 提取原子特征（原子类型、度、电荷、芳香性等）
- **CanonicalBondFeaturizer**: 提取键特征（键类型、是否共轭、是否在环中等）
- **mol_to_bigraph**: 将 RDKit 分子对象转为 DGL 双向图（加自环）

### 源码对应
- 文件: `scripts/dataset.py` → `USPTODataset._pre_process()`
- 文件: `scripts/utils.py` → `init_featurizer()`
- 特征化器: `dgllife.utils.WeaveAtomFeaturizer`, `CanonicalBondFeaturizer`

In [10]:
# ====== Step 4.1: 初始化特征化器 ======
# 源码对应: scripts/utils.py → init_featurizer()

from dgllife.utils import WeaveAtomFeaturizer, CanonicalBondFeaturizer, mol_to_bigraph
from functools import partial

# LocalTransform 使用的 64 种原子类型
atom_types = ['C', 'N', 'O', 'S', 'F', 'Si', 'P', 'Cl', 'Br', 'Mg', 'Na', 'Ca', 'Fe',
         'As', 'Al', 'I', 'B', 'V', 'K', 'Tl', 'Yb', 'Sb', 'Sn', 'Ag', 'Pd', 'Co', 'Se', 'Ti',
         'Zn', 'H', 'Li', 'Ge', 'Cu', 'Au', 'Ni', 'Cd', 'In', 'Mn', 'Zr', 'Cr', 'Pt', 'Hg', 'Pb',
         'W', 'Ru', 'Nb', 'Re', 'Te', 'Rh', 'Ta', 'Tc', 'Ba', 'Bi', 'Hf', 'Mo', 'U', 'Sm', 'Os', 'Ir',
         'Ce', 'Gd', 'Ga', 'Cs']

node_featurizer = WeaveAtomFeaturizer(atom_types=atom_types)
edge_featurizer = CanonicalBondFeaturizer(self_loop=True)

print(f'原子类型数: {len(atom_types)}')
print(f'节点特征维度: {node_featurizer.feat_size()}')
print(f'边特征维度: {edge_featurizer.feat_size()}')

/home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/envs/localtransform_tutorial_envs/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


原子类型数: 63
节点特征维度: 80
边特征维度: 13


In [11]:
# ====== Step 4.2: 构建特征图 ======
# 源码对应: scripts/dataset.py → USPTODataset._pre_process() 中的 mol_to_graph()

import torch
import dgl

smiles = 'CC(=O)Cl'  # 乙酰氯
mol = Chem.MolFromSmiles(smiles)

# 创建 DGL 双向图（加自环）
fgraph = mol_to_bigraph(
    mol,
    node_featurizer=node_featurizer,
    edge_featurizer=edge_featurizer,
    add_self_loop=True,
    canonical_atom_order=False  # 保持原始原子顺序（重要！）
)

print(f'分子: {smiles}')
print(f'原子: {[(a.GetIdx(), a.GetSymbol()) for a in mol.GetAtoms()]}')
print(f'\nDGL 特征图:')
print(f'  节点数: {fgraph.num_nodes()} (= 原子数)')
print(f'  边数: {fgraph.num_edges()} (= 键数×2 + 自环)')
print(f'  节点特征 shape: {fgraph.ndata["h"].shape}')
print(f'  边特征 shape: {fgraph.edata["e"].shape}')

print(f'\n节点特征 (h) — 每个原子的化学特征向量:')
for i in range(fgraph.num_nodes()):
    feat = fgraph.ndata['h'][i]
    symbol = mol.GetAtomWithIdx(i).GetSymbol()
    print(f'  原子 {i} ({symbol}): shape={feat.shape}, 前10维={feat[:10].tolist()}')

分子: CC(=O)Cl
原子: [(0, 'C'), (1, 'C'), (2, 'O'), (3, 'Cl')]

DGL 特征图:
  节点数: 4 (= 原子数)
  边数: 10 (= 键数×2 + 自环)
  节点特征 shape: torch.Size([4, 80])
  边特征 shape: torch.Size([10, 13])

节点特征 (h) — 每个原子的化学特征向量:
  原子 0 (C): shape=torch.Size([80]), 前10维=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
  原子 1 (C): shape=torch.Size([80]), 前10维=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
  原子 2 (O): shape=torch.Size([80]), 前10维=[0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
  原子 3 (Cl): shape=torch.Size([80]), 前10维=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0]


In [12]:
# ====== Step 4.3: 解读原子特征的含义 ======

# WeaveAtomFeaturizer 的特征分解
print('=== WeaveAtomFeaturizer 特征构成 ===')
print(f'总维度: {node_featurizer.feat_size()}')
print()
print('各子特征说明:')
feature_info = [
    ('原子类型 one-hot', len(atom_types), '64 种原子类型的独热编码'),
    ('手性 one-hot', 2, 'R/S 手性'),
    ('度 one-hot', 7, '连接的键数 (0-6)'),
    ('电荷', 1, '形式电荷'),
    ('氢数 one-hot', 5, '连接的氢数 (0-4)'),
    ('杂化 one-hot', 3, 'sp, sp2, sp3'),
    ('芳香性', 1, '是否在芳香环中'),
]
total = 0
for name, dim, desc in feature_info:
    print(f'  [{total}:{total+dim}] {name} (dim={dim}) — {desc}')
    total += dim

print(f'\n> 注意: 实际维度可能略有不同（取决于 dgllife 版本），以 feat_size() 返回值为准')

print(f'\n=== CanonicalBondFeaturizer 特征构成 ===')
print(f'总维度: {edge_featurizer.feat_size()}')
print('各子特征: 键类型(4) + 共轭(1) + 在环中(1) + 立体(5) = ~12 维')
print('(自环边用零向量表示)')

=== WeaveAtomFeaturizer 特征构成 ===
总维度: 80

各子特征说明:
  [0:64] 原子类型 one-hot (dim=64) — 64 种原子类型的独热编码
  [64:66] 手性 one-hot (dim=2) — R/S 手性
  [66:73] 度 one-hot (dim=7) — 连接的键数 (0-6)
  [73:74] 电荷 (dim=1) — 形式电荷
  [74:79] 氢数 one-hot (dim=5) — 连接的氢数 (0-4)
  [79:82] 杂化 one-hot (dim=3) — sp, sp2, sp3
  [82:83] 芳香性 (dim=1) — 是否在芳香环中

> 注意: 实际维度可能略有不同（取决于 dgllife 版本），以 feat_size() 返回值为准

=== CanonicalBondFeaturizer 特征构成 ===
总维度: 13
各子特征: 键类型(4) + 共轭(1) + 在环中(1) + 立体(5) = ~12 维
(自环边用零向量表示)


## 5. Step 4: 分子图构建 — 稠密图 (Dense Graph)

### 为什么还需要稠密图？

特征图只包含分子的**拓扑连接信息**。但 LocalTransform 的注意力机制还需要：
1. **原子距离矩阵 (ADM)**: 所有原子对之间的拓扑距离，用作注意力的位置编码
2. **虚拟键列表 + 真实键列表**: 模型需要对每条键做预测

### 两种图的区别

| 维度 | 特征图 (Feature Graph) | 稠密图 (Dense Graph) |
|------|----------------------|---------------------|
| 类型 | DGL 图 | 字典（矩阵+列表） |
| 节点 | 原子 + 特征向量 | — |
| 边 | 化学键 + 特征向量 | — |
| 距离 | — | 原子距离矩阵 |
| 键列表 | — | 虚拟键 + 真实键 |
| 用途 | MPNN 消息传递 | 注意力位置编码 + 键级别预测 |

### 源码对应
- 文件: `scripts/dataset.py` → `get_adm()`, `get_bonds()`

In [13]:
# ====== Step 5.1: 构建原子距离矩阵 ======
# 源码对应: scripts/dataset.py → get_adm()

def get_adm(mol, max_distance=6):
    """计算原子距离矩阵
    
    Args:
        mol: RDKit 分子对象
        max_distance: 最大距离截断值
    Returns:
        distance_matrix: 形状为 (num_atoms, num_atoms) 的距离矩阵
            值域: 0 ~ max_distance (同分子), max_distance+1 (不同分子)
    """
    mol_size = mol.GetNumAtoms()
    distance_matrix = np.ones((mol_size, mol_size)) * (max_distance + 1)
    dm = Chem.GetDistanceMatrix(mol)
    dm[dm > 100] = -1                    # 不同分子片段 → 标记为 -1
    dm[dm > max_distance] = max_distance  # 同分子内远距离 → 截断
    dm[dm == -1] = max_distance + 1       # 不同分子 → max+1
    distance_matrix[:dm.shape[0], :dm.shape[1]] = dm
    return distance_matrix

# 用多组分反应物演示
smiles = 'CC(=O)Cl.NCC'  # 乙酰氯.甲胺 (两个分子)
mol = Chem.MolFromSmiles(smiles)

atoms = [(a.GetIdx(), a.GetSymbol()) for a in mol.GetAtoms()]
print(f'分子: {smiles}')
print(f'原子列表: {atoms}')

adm = get_adm(mol)
print(f'\n原子距离矩阵 (ADM):')
print(f'  shape: {adm.shape}')

# 格式化打印
header = '     ' + '  '.join([f'{a[1]:>3}' for a in atoms])
print(header)
for i, (idx, sym) in enumerate(atoms):
    row = f'{sym}({idx})' + '  '.join([f'{int(adm[i][j]):>3}' for j in range(len(atoms))])
    print(row)

print(f'\n> 0 = 自身, 1-{6} = 拓扑距离, {7} = 不同分子片段')
print(f'> 注意 CC(=O)Cl 和 NCC 之间的距离为 7（不同分子）')

分子: CC(=O)Cl.NCC
原子列表: [(0, 'C'), (1, 'C'), (2, 'O'), (3, 'Cl'), (4, 'N'), (5, 'C'), (6, 'C')]

原子距离矩阵 (ADM):
  shape: (7, 7)
       C    C    O   Cl    N    C    C
C(0)  0    1    2    2    7    7    7
C(1)  1    0    1    1    7    7    7
O(2)  2    1    0    2    7    7    7
Cl(3)  2    1    2    0    7    7    7
N(4)  7    7    7    7    0    1    2
C(5)  7    7    7    7    1    0    1
C(6)  7    7    7    7    2    1    0

> 0 = 自身, 1-6 = 拓扑距离, 7 = 不同分子片段
> 注意 CC(=O)Cl 和 NCC 之间的距离为 7（不同分子）


In [15]:
# ====== Step 5.2: 构建虚拟键和真实键列表 ======
# 源码对应: scripts/dataset.py → get_bonds()

def get_bonds(smiles):
    """获取分子的虚拟键和真实键
    
    Returns:
        V: numpy array, 虚拟键 (未成键的原子对)
        B: numpy array, 真实键 (已有的化学键)
    """
    mol = Chem.MolFromSmiles(smiles)
    A = list(range(mol.GetNumAtoms()))
    B = []
    for atom in mol.GetAtoms():
        others = []
        for bond in atom.GetBonds():
            atoms_pair = [bond.GetBeginAtom().GetIdx(), bond.GetEndAtom().GetIdx()]
            other = [a for a in atoms_pair if a != atom.GetIdx()][0]
            others.append(other)
        B += [(atom.GetIdx(), other) for other in sorted(others)]
    V = [(a, b) for a in A for b in A if a != b and (a, b) not in B]
    return np.array(V), np.array(B)

# 演示
smiles_sep = 'CC(=O)Cl'  # 仅反应物（-sep 模式）
v_bonds, r_bonds = get_bonds(smiles_sep)

mol = Chem.MolFromSmiles(smiles_sep)
print(f'分子: {smiles_sep}')
print(f'原子: {[(a.GetIdx(), a.GetSymbol()) for a in mol.GetAtoms()]}')
print(f'\n真实键 (Real): {r_bonds.shape}')
for i, (a, b) in enumerate(r_bonds):
    a, b = int(a), int(b)
    sym_a = mol.GetAtomWithIdx(a).GetSymbol()
    sym_b = mol.GetAtomWithIdx(b).GetSymbol()
    print(f'  [{i}] {sym_a}({a}) — {sym_b}({b})')

print(f'\n虚拟键 (Virtual): {v_bonds.shape}')
for i, (a, b) in enumerate(v_bonds[:8]):
    a, b = int(a), int(b)
    sym_a = mol.GetAtomWithIdx(a).GetSymbol()
    sym_b = mol.GetAtomWithIdx(b).GetSymbol()
    print(f'  [{i}] {sym_a}({a}) ··· {sym_b}({b})')
if len(v_bonds) > 8:
    print(f'  ... (共 {len(v_bonds)} 条)')

分子: CC(=O)Cl
原子: [(0, 'C'), (1, 'C'), (2, 'O'), (3, 'Cl')]

真实键 (Real): (6, 2)
  [0] C(0) — C(1)
  [1] C(1) — C(0)
  [2] C(1) — O(2)
  [3] C(1) — Cl(3)
  [4] O(2) — C(1)
  [5] Cl(3) — C(1)

虚拟键 (Virtual): (6, 2)
  [0] C(0) ··· O(2)
  [1] C(0) ··· Cl(3)
  [2] O(2) ··· C(0)
  [3] O(2) ··· Cl(3)
  [4] Cl(3) ··· C(0)
  [5] Cl(3) ··· O(2)


In [16]:
# ====== Step 5.3: 组合成完整的稠密图 ======
# 源码对应: scripts/dataset.py → USPTODataset._pre_process()

smiles_full = 'CC(=O)Cl.NCC'  # 完整反应物
smiles_sep = 'CC(=O)Cl'       # 仅主反应物（用于提取键列表）

mol_full = Chem.MolFromSmiles(smiles_full)

# 构建稠密图
dgraph = {
    'atom_distance_matrix': get_adm(mol_full),  # ADM 用完整分子
    'bonds': get_bonds(smiles_sep),              # 键列表用主反应物（-sep 模式）
}
dgraph['v_bonds'] = dgraph['bonds'][0]  # 虚拟键
dgraph['r_bonds'] = dgraph['bonds'][1]  # 真实键

print('稠密图结构:')
print(f'  atom_distance_matrix: shape={dgraph["atom_distance_matrix"].shape}')
print(f'  v_bonds (虚拟键): shape={dgraph["v_bonds"].shape}')
print(f'  r_bonds (真实键): shape={dgraph["r_bonds"].shape}')

print(f'\n> -sep 模式: ADM 用完整反应物+试剂，但键列表只用主反应物')
print(f'> -mix 模式: 两者都用完整反应物+试剂')
print(f'> 教学中使用的预训练模型为 mix 模式 (LocalTransform_mix.pth)')

稠密图结构:
  atom_distance_matrix: shape=(7, 7)
  v_bonds (虚拟键): shape=(6, 2)
  r_bonds (真实键): shape=(6, 2)

> -sep 模式: ADM 用完整反应物+试剂，但键列表只用主反应物
> -mix 模式: 两者都用完整反应物+试剂
> 教学中使用的预训练模型为 mix 模式 (LocalTransform_mix.pth)


## 6. Step 5: 批处理与数据整理 (Collation)

### 为什么需要数据整理？

不同分子的原子数、键数不同，无法直接 stack 成张量。需要：
1. **DGL batch**: 将多个图合并成一个大图
2. **ADM padding**: 将不同大小的距离矩阵填充到同一尺寸
3. **键列表汇总**: 将每个分子的键列表组合成字典

### 源码对应
- 文件: `scripts/utils.py` → `collate_molgraphs()`, `pad_atom_distance_matrix()`

In [17]:
# ====== Step 6.1: DGL 图的批处理 ======
import torch
import dgl

# 准备两个不同大小的分子
smiles_list = ['CCO', 'c1ccccc1O']  # 乙醇, 苯酚

graphs = []
for s in smiles_list:
    mol = Chem.MolFromSmiles(s)
    g = mol_to_bigraph(mol, node_featurizer=node_featurizer, 
                       edge_featurizer=edge_featurizer,
                       add_self_loop=True, canonical_atom_order=False)
    graphs.append(g)
    print(f'{s}: {g.num_nodes()} 节点, {g.num_edges()} 边')

# DGL batch: 合并为一个大图
bg = dgl.batch(graphs)
print(f'\n批处理后:')
print(f'  总节点数: {bg.num_nodes()} (= {" + ".join([str(g.num_nodes()) for g in graphs])})')
print(f'  总边数: {bg.num_edges()}')
print(f'  节点特征: {bg.ndata["h"].shape}')
print(f'  边特征: {bg.edata["e"].shape}')

# 可以拆回来
unbatched = dgl.unbatch(bg)
print(f'\n拆分后: {[g.num_nodes() for g in unbatched]} 个节点')
print('\n> DGL batch 将多个图合并为一个大图，节点和边连续排列')
print('> 每个子图互不连接，unbatch 可以恢复')

CCO: 3 节点, 7 边
c1ccccc1O: 7 节点, 21 边

批处理后:
  总节点数: 10 (= 3 + 7)
  总边数: 28
  节点特征: torch.Size([10, 80])
  边特征: torch.Size([28, 13])

拆分后: [3, 7] 个节点

> DGL batch 将多个图合并为一个大图，节点和边连续排列
> 每个子图互不连接，unbatch 可以恢复


In [18]:
# ====== Step 6.2: 原子距离矩阵的批处理 (Padding) ======
# 源码对应: scripts/utils.py → pad_atom_distance_matrix()

def pad_atom_distance_matrix(adm_list):
    """将不同大小的 ADM 填充到同一尺寸
    
    Args:
        adm_list: 多个 ADM numpy 数组的列表
    Returns:
        padded: (batch_size, max_atoms, max_atoms) 张量
    """
    max_size = max([adm.shape[0] for adm in adm_list])
    padded = []
    for adm in adm_list:
        pad_size = max_size - adm.shape[0]
        padded_adm = np.pad(adm, (0, pad_size), 'maximum')  # 用最大值填充
        padded.append(torch.tensor(padded_adm).unsqueeze(0).long())
    return torch.cat(padded, dim=0)

# 演示
adm_list = [
    get_adm(Chem.MolFromSmiles('CCO')),       # 3×3
    get_adm(Chem.MolFromSmiles('c1ccccc1O')),  # 7×7
]

print(f'填充前:')
for i, adm in enumerate(adm_list):
    print(f'  分子 {i}: ADM shape = {adm.shape}')

padded = pad_atom_distance_matrix(adm_list)
print(f'\n填充后: shape = {padded.shape}')
print(f'  dtype = {padded.dtype}')
print(f'\n分子 0 的填充矩阵 (前3×3为有效区域):')
print(padded[0].numpy())
print(f'\n> 填充区域使用最大距离值 (7)，模型通过 mask 忽略')

填充前:
  分子 0: ADM shape = (3, 3)
  分子 1: ADM shape = (7, 7)

填充后: shape = torch.Size([2, 7, 7])
  dtype = torch.int64

分子 0 的填充矩阵 (前3×3为有效区域):
[[0 1 2 2 2 2 2]
 [1 0 1 1 1 1 1]
 [2 1 0 2 2 2 2]
 [2 1 2 2 2 2 2]
 [2 1 2 2 2 2 2]
 [2 1 2 2 2 2 2]
 [2 1 2 2 2 2 2]]

> 填充区域使用最大距离值 (7)，模型通过 mask 忽略


In [19]:
# ====== Step 6.3: 完整的数据整理流程 ======
# 源码对应: scripts/utils.py → collate_molgraphs()

# 模拟一个 mini-batch 的完整数据整理
smiles_batch = ['CC(=O)Cl', 'c1ccccc1O']

fgraphs = []
dgraphs = []

for s in smiles_batch:
    mol = Chem.MolFromSmiles(s)
    # 特征图
    fg = mol_to_bigraph(mol, node_featurizer=node_featurizer,
                        edge_featurizer=edge_featurizer,
                        add_self_loop=True, canonical_atom_order=False)
    fgraphs.append(fg)
    
    # 稠密图
    v, r = get_bonds(s)
    dg = {
        'atom_distance_matrix': get_adm(mol),
        'v_bonds': v,
        'r_bonds': r,
    }
    dgraphs.append(dg)

# 1. 批处理特征图
bg = dgl.batch(fgraphs)

# 2. 填充距离矩阵
adms = pad_atom_distance_matrix([dg['atom_distance_matrix'] for dg in dgraphs])

# 3. 组织键列表
bonds_dicts = {
    'virtual': [torch.from_numpy(dg['v_bonds']).long() for dg in dgraphs],
    'real': [torch.from_numpy(dg['r_bonds']).long() for dg in dgraphs],
}

print('=== 完整批处理结果 ===')
print(f'\n1. 批处理特征图 (bg):')
print(f'   节点数: {bg.num_nodes()}, 边数: {bg.num_edges()}')
print(f'   节点特征: {bg.ndata["h"].shape}')
print(f'   边特征: {bg.edata["e"].shape}')

print(f'\n2. 填充后的距离矩阵 (adms):')
print(f'   shape: {adms.shape}  (batch_size × max_atoms × max_atoms)')

print(f'\n3. 键列表字典 (bonds_dicts):')
for bond_type in ['virtual', 'real']:
    shapes = [str(b.shape) for b in bonds_dicts[bond_type]]
    print(f'   {bond_type}: {shapes}')

print(f'\n> 这就是模型 forward() 接收的三个核心输入！')
print(f'> model(bg, adms, bonds_dicts, node_feats, edge_feats)')

=== 完整批处理结果 ===

1. 批处理特征图 (bg):
   节点数: 11, 边数: 31
   节点特征: torch.Size([11, 80])
   边特征: torch.Size([31, 13])

2. 填充后的距离矩阵 (adms):
   shape: torch.Size([2, 7, 7])  (batch_size × max_atoms × max_atoms)

3. 键列表字典 (bonds_dicts):
   virtual: ['torch.Size([6, 2])', 'torch.Size([28, 2])']
   real: ['torch.Size([6, 2])', 'torch.Size([14, 2])']

> 这就是模型 forward() 接收的三个核心输入！
> model(bg, adms, bonds_dicts, node_feats, edge_feats)


## 7. 补充: 标签的整理过程

### 为什么单独讲标签整理？

训练时标签需要与键列表对齐。每条键的标签存储在一个列表中，非反应键标签为 0，反应键标签为模板类别 ID。

### 源码对应
- 文件: `scripts/utils.py` → `make_labels()`, `get_reactive_template_labels()`

In [20]:
# ====== Step 7.1: 标签整理示例 ======
# 源码对应: scripts/utils.py → make_labels()

# 模拟一个样本的标签（来自 labeled_data.csv）
# 格式: [(键类型, 键索引, 模板类别), ...]
sample_labels = [('v', 5, 2535), ('r', 2, 4370)]  # 真实标签示例

# 假设我们有 10 条虚拟键和 6 条真实键
n_virtual = 10
n_real = 6

# 初始化全 0 标签
vtemplate_label = [0] * n_virtual
rtemplate_label = [0] * n_real

# 分配标签
for label in sample_labels:
    edit_type, edit_idx, edit_template = label
    if edit_type == 'v':
        vtemplate_label[edit_idx] = edit_template
    else:
        rtemplate_label[edit_idx] = edit_template

print('标签分配结果:')
print(f'  虚拟键标签: {vtemplate_label}')
print(f'  真实键标签: {rtemplate_label}')
print(f'\n> 绝大多数位置为 0 (不参与反应)，只有少数位置有非零模板类别')
print(f'> 这种极度不平衡促使 LocalTransform 使用加权损失: class_0 权重=0.5, 其余=1.0')

# 反应性标签（二值化）
v_reactive = [int(l > 0) for l in vtemplate_label]
r_reactive = [int(l > 0) for l in rtemplate_label]
print(f'\n反应性标签 (binary):')
print(f'  虚拟键: {v_reactive}')
print(f'  真实键: {r_reactive}')
print(f'\n> 模型有两个损失: 反应性损失 (R_loss) + 模板分类损失 (T_loss)')

标签分配结果:
  虚拟键标签: [0, 0, 0, 0, 0, 2535, 0, 0, 0, 0]
  真实键标签: [0, 0, 4370, 0, 0, 0]

> 绝大多数位置为 0 (不参与反应)，只有少数位置有非零模板类别
> 这种极度不平衡促使 LocalTransform 使用加权损失: class_0 权重=0.5, 其余=1.0

反应性标签 (binary):
  虚拟键: [0, 0, 0, 0, 0, 1, 0, 0, 0, 0]
  真实键: [0, 0, 1, 0, 0, 0]

> 模型有两个损失: 反应性损失 (R_loss) + 模板分类损失 (T_loss)


## 总结

### 本 notebook 完成了什么

| 步骤 | 内容 | 源码文件 |
|------|------|----------|
| 1 | 了解原始数据格式和整体数据流 | — |
| 2 | 反应模板提取：SMILES → 局部 SMARTS 模板 | `preprocessing/Extract_from_train_data.py` |
| 3 | 反应标注：为每条键分配模板类别标签 | `preprocessing/Run_preprocessing.py` |
| 4 | 特征图构建：SMILES → DGL 图 (原子/键特征) | `scripts/dataset.py`, `scripts/utils.py` |
| 5 | 稠密图构建：原子距离矩阵 + 键列表 | `scripts/dataset.py` |
| 6 | 批处理：图 batch + ADM padding + 键列表汇总 | `scripts/utils.py` |
| 7 | 标签整理：sparse 标签 → 逐键标签向量 | `scripts/utils.py` |

### 关键数据结构回顾

| 数据 | 类型 | 说明 |
|------|------|------|
| `fgraph` | DGL 图 | 节点特征 `h` + 边特征 `e` |
| `adm` | (N, N) numpy | 拓扑距离矩阵，截断到 6 |
| `v_bonds` | (M, 2) numpy | 虚拟键原子对 |
| `r_bonds` | (K, 2) numpy | 真实键原子对 |
| `labels` | [(type, idx, class), ...] | 键级别模板标签 |

### 教学版与生产版差异

| 方面 | 原仓库 | 教学版 |
|------|--------|--------|
| 数据量 | 480k 反应 | 1-2 个示例 |
| 缓存 | 图序列化到磁盘 (.bin/.pkl) | 不缓存 |
| 并行 | DataLoader num_workers | 串行处理 |
| 模板提取 | 全量训练集 | 使用预提取的模板 |

### 下一步

→ **Notebook 3: 模型展示** — 了解 LocalTransform 的模型架构和推理流程